In [ ]:
import json
import joblib
import numpy as np
from kafka import KafkaConsumer, KafkaProducer

KAFKA_BOOTSTRAP_SERVERS = ["localhost:9092"]
INPUT_TOPIC  = "water_quality_final"
OUTPUT_TOPIC = "ml_predictions_final"
GROUP_ID     = "water_quality_processors"
MODEL_PATH   = "aeration_classifier.joblib"

print(f"Loading model from '{MODEL_PATH}' ...")
model = joblib.load(MODEL_PATH)
print("Model loaded successfully.")

consumer = KafkaConsumer(
    INPUT_TOPIC,
    bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
    value_deserializer=lambda m: json.loads(m.decode("utf-8")),
    auto_offset_reset="latest",
    enable_auto_commit=True,
    group_id=GROUP_ID,
)

producer = KafkaProducer(
    bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
)

print(f"Subscribed to '{INPUT_TOPIC}'. Publishing results to '{OUTPUT_TOPIC}'.")
print("Waiting for messages … (Ctrl+C to stop)\n")

try:
    for message in consumer:
        sensor_data = message.value
        sensor_key = message.key
        
        features = np.array([[
            sensor_data["water_temperature"],
            sensor_data["ph_level"],
            sensor_data["turbidity"],
            sensor_data["dissolved_oxygen"],
        ]])

        prediction   = int(model.predict(features)[0])        
        probabilities = model.predict_proba(features)[0].tolist() 

        sensor_key_str = sensor_key.decode('utf-8') if sensor_key else None

        result = {
            "sensor_id": sensor_key_str, 
            "timestamp": sensor_data["timestamp"],
            "prediction": prediction,                
            "prob_class0": round(probabilities[0], 4),
            "prob_class1": round(probabilities[1], 4),
            "water_temperature": sensor_data["water_temperature"],
            "ph_level": sensor_data["ph_level"],
            "turbidity": sensor_data["turbidity"],
            "dissolved_oxygen": sensor_data["dissolved_oxygen"],
        }

        inference_summary = {
            "sensor_id": str(sensor_data.get("sensor_id", "?")),
            "timestamp": sensor_data["timestamp"],
            "prediction_results": [round(probabilities[0], 1), round(probabilities[1], 1)],
        }
        print("forming inference")
        print(f"t: {inference_summary}")
        print(f"Received from sensor {sensor_data.get('sensor_id', '?')}: {sensor_data}")

        producer.send(OUTPUT_TOPIC, value=result, key=sensor_key)
        producer.flush()

except KeyboardInterrupt:
    print("\nStopped ML processor.")

finally:
    consumer.close()
    producer.close()

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.4.0 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Loading model from 'aeration_classifier.joblib' ...


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.4.0 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Model loaded successfully.
Subscribed to 'water_quality_final'. Publishing results to 'ml_predictions_final'.
Waiting for messages … (Ctrl+C to stop)

